# Legacy notebook — use train_kira_full.ipynb for "key kira"

This notebook prepares a Kira-specific copy of the 2026-compatible openWakeWord training notebook.
It uses **hey Kira / okay Kira / hello Kira** rather than bare `Kira` to reduce false accepts.

**Do not run this legacy generator.** Open `train_kira_full.ipynb` directly in Colab, choose a **T4 GPU**, then use **Runtime → Run all**.


In [ ]:
import subprocess, pathlib, shutil
subprocess.run(['git','clone','--depth','1','https://github.com/saintpete/openwakeword-colab-toolkit.git'], check=True)
print('Toolkit cloned.')


In [ ]:
import json, pathlib
src = pathlib.Path('openwakeword-colab-toolkit/hey_clem_colab.ipynb')
nb = json.loads(src.read_text())

# Carefully replace model/phrase-specific strings while retaining all 2026 compatibility fixes.
replacements = [
 ('hey_clem_outputs', 'kira_outputs'),
 ('hey_clem_model', 'kira_model'),
 ('hey_clem.yaml', 'kira.yaml'),
 ('hey_clem.onnx', 'kira.onnx'),
 ('hey_clem', 'kira'),
 ('hey Clem', 'hey Kira'), ('Hey Clem', 'Hey Kira'),
 ('okay Clem', 'okay Kira'), ('Okay Clem', 'Okay Kira'),
 ('hello Clem', 'hello Kira'), ('Hello Clem', 'Hello Kira'),
 ('clem', 'kira'), ('Clem', 'Kira'),
]
for cell in nb['cells']:
    text = ''.join(cell.get('source', []))
    for a,b in replacements:
        text = text.replace(a,b)
    cell['source'] = text.splitlines(keepends=True)

# Replace the config cell with a clean Kira config, avoiding Clem-specific negatives/comments.
for cell in nb['cells']:
    text = ''.join(cell.get('source', []))
    if '%%writefile kira.yaml' in text:
        cfg = '''%%writefile kira.yaml
model_name: "kira"
target_phrase:
  - "hey kira"
  - "okay kira"
  - "hello kira"
custom_negative_phrases:
  - "kira"
  - "hey kiraa"
  - "hey kiera"
  - "hey keira"
  - "hey kira"
  - "hey vera"
  - "hey mira"
  - "hey zira"
  - "hey lira"
  - "hey sierra"
  - "okay mira"
  - "hello vera"
n_samples: 75000
n_samples_val: 7500
tts_batch_size: 50
augmentation_batch_size: 16
augmentation_rounds: 2
piper_sample_generator_path: "./piper-sample-generator"
output_dir: "./kira_model"
rir_paths:
  - "./mit_rirs"
background_paths:
  - "./audioset_16k"
  - "./fma"
background_paths_duplication_rate:
  - 1
  - 1
false_positive_validation_data_path: "./validation_set_features.npy"
feature_data_files:
  "ACAV100M_sample": "./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
batch_n_per_class:
  "ACAV100M_sample": 1024
  "adversarial_negative": 50
  "positive": 50
model_type: "dnn"
layer_size: 32
steps: 50000
max_negative_weight: 1500
target_false_positives_per_hour: 0.2
'''
        # Remove accidental exact-target negative after writing; keep near misses only.
        cfg = cfg.replace('  - "hey kira"\n  - "hey vera"', '  - "hey kira please"\n  - "hey vera"')
        cell['source'] = cfg.splitlines(keepends=True)
        break

# Fix preflight phrase/assertion so it does not require Clem-specific phonemes.
for cell in nb['cells']:
    text = ''.join(cell.get('source', []))
    text = text.replace('phonemize("hello kira")', 'phonemize("hello kira")')
    text = text.replace('assert "kl" in ipa, f"unexpected phonemes: {ipa}"', 'assert ipa, f"empty phonemes: {ipa}"')
    cell['source'] = text.splitlines(keepends=True)

out = pathlib.Path('/content/train_kira_full.ipynb')
out.write_text(json.dumps(nb, ensure_ascii=False, indent=1))
print('Created:', out)


In [ ]:
from google.colab import files
files.download('/content/train_kira_full.ipynb')
print('Open train_kira_full.ipynb in Colab, select T4 GPU, then Runtime -> Run all.')
